In [11]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
!pip install unsloth
!pip install xformers trl peft accelerate bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/

In [2]:
from datasets import load_dataset

dataset = load_dataset("viv2005ek/TextRewriterInTonesAndGrammer", split="train")
print(dataset[0])  # Check structure

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

Dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/571 [00:00<?, ? examples/s]

{'instruction': 'Rewrite the following text in a professional style.', 'input': 'hey can u pls check this file asap, its kinda urgent', 'output': 'Could you please review this file at your earliest convenience? It is urgent.'}


In [3]:
from datasets import load_dataset

dataset = load_dataset("viv2005ek/TextRewriterInTonesAndGrammer", split="train")
print(dataset[0])  # Check structure

{'instruction': 'Rewrite the following text in a professional style.', 'input': 'hey can u pls check this file asap, its kinda urgent', 'output': 'Could you please review this file at your earliest convenience? It is urgent.'}


In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-3-mini-4k-instruct",  # Good balance of size & quality
    max_seq_length=2048,
    dtype=None,  # auto
    load_in_4bit=True,  # 4-bit quantization for memory efficiency
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_func(example):
    return [f"<|user|>\n{example['instruction']}\n{example['input']}\n<|assistant|>\n{example['output']}"]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./rewriter_model",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        push_to_hub=True,
        hub_model_id="viv2005ek/phi-rewriter",
    ),
    formatting_func=formatting_func,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,582,912 of 3,833,662,464 (0.33% trained)


Step,Training Loss


TrainOutput(global_step=3, training_loss=0.13783633708953857, metrics={'train_runtime': 33.7204, 'train_samples_per_second': 0.534, 'train_steps_per_second': 0.089, 'total_flos': 413079019978752.0, 'train_loss': 0.13783633708953857, 'epoch': 3.0})

In [18]:
from huggingface_hub import HfApi

# Upload the entire checkpoint folder
api = HfApi()
api.upload_folder(
    folder_path="./rewriter_model/checkpoint-3",   # the path to your final checkpoint
    repo_id="viv2005ek/phi-rewriter-adapter",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...point-3/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ...adapter_model.safetensors: 100%|##########| 50.4MB / 50.4MB            

  ...checkpoint-3/optimizer.pt:   1%|          |  551kB /  101MB            

  ...heckpoint-3/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...el/checkpoint-3/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

  ...checkpoint-3/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

CommitInfo(commit_url='https://huggingface.co/viv2005ek/phi-rewriter-adapter/commit/ad62e7de84fb69253912db64ddfcfeedd2d147e8', commit_message='Upload folder using huggingface_hub', commit_description='', oid='ad62e7de84fb69253912db64ddfcfeedd2d147e8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/viv2005ek/phi-rewriter-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='viv2005ek/phi-rewriter-adapter'), pr_revision=None, pr_num=None)

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "microsoft/Phi-3-mini-4k-instruct"
adapter_repo = "viv2005ek/phi-rewriter-adapter"

# Load base model (4-bit quantization optional, but you can also load in fp16)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_repo)

# Test with a sample


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/50.4M [00:00<?, ?B/s]

Subject: Request for Report Submission by End of Day

Dear [Recipient's Name],

I hope this message finds you well. I am writing to kindly request the submission of the report that was assigned to you. As per our project timeline, we would appreciate receiving the report by the end of the day (EOD).

Thank you for your attention to this matter.

Best regards,
[Your Name]


Text


In [36]:
def rewrite(text, style):
    style_map = {
        "professional": "Rewrite the following text in a professional style.",
        "slangy": "Rewrite the following text in a slangy style.",
        "human": "Rewrite the following text to sound human and natural.",
        "grammar": "Rewrite the following text to correct grammar errors."
    }
    instruction = style_map[style]
    prompt = f"<|user|>\n{instruction}\n{text}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,           # enough for a detailed rewrite
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )

    # Decode only the newly generated part
    generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # 1. Remove any trailing content after a new instruction pattern
    stop_patterns = [
        "Rewrite the following text",
        "---",
        "Original:",
        "Text:",
        "Revised Text:",
        "Corrected:",
        "**",
        "\n\n\n"  # triple newline often separates unrelated blocks
    ]
    stop_idx = len(generated)
    for pat in stop_patterns:
        idx = generated.find(pat)
        if idx != -1 and idx < stop_idx:
            stop_idx = idx
    generated = generated[:stop_idx]

    # 2. If there are multiple sections separated by double newline,
    #    decide whether to keep all or only the first.
    sections = generated.split("\n\n")
    if len(sections) > 1:
        # Check if the first section looks like a complete rewrite
        first = sections[0].strip()
        # If the first section is very short (like a single sentence) and
        # the second section starts with a capital letter and is a full sentence,
        # it's likely a list of alternatives. Keep only the first.
        if len(first) < 200 and any(sec.strip() and sec[0].isupper() for sec in sections[1:]):
            generated = first
        else:
            # Otherwise, keep all sections (they are probably part of one rewrite)
            generated = "\n\n".join(sections)

    # 3. If the result still contains obvious extra markers (like "I'm doing well"),
    #    we can apply a final safety cut.
    extra_cut = [
        "I'm doing well",
        "I'm good too",
        "Looking forward to it",
        "Me too, see you soon"
    ]
    for marker in extra_cut:
        idx = generated.find(marker)
        if idx != -1:
            generated = generated[:idx]
            break

    # Clean up trailing whitespace and ensure we have something
    generated = generated.strip()
    if not generated:
        generated = "[Unable to rewrite]"

    return generated

In [1]:
# 1. Install necessary libraries (if not already installed)
!pip install -q transformers peft accelerate bitsandbytes

# 2. Import required modules
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 3. Define your base model and the adapter's path
base_model_name = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "viv2005ek/phi-rewriter-adapter" # Your adapter's ID on the Hub

# 4. Load the base model
print(f"Loading base model: {base_model_name}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,  # Use half-precision to save memory
    device_map="auto"            # Automatically place layers on available devices
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# 5. Load the fine-tuned adapter on top of the base model
print(f"Loading adapter: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 6. Merge the adapter with the base model
print("Merging the adapter...")
merged_model = model.merge_and_unload()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
Loading base model: microsoft/Phi-3-mini-4k-instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/50.4M [00:00<?, ?B/s]

Merging the adapter...
Saving merged model to ./phi-rewriter-merged


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ImportError: cannot import name 'Repository' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [3]:

# 7. Save the merged model locally
output_dir = "./phi-rewriter-merged"
print(f"Saving merged model to {output_dir}")
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)




Saving merged model to ./phi-rewriter-merged


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ImportError: cannot import name 'Repository' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [6]:
api = HfApi()
api.upload_folder(
    folder_path='./phi-rewriter-merged',
    repo_id="viv2005ek/phi-rewriter", # The target repo for your merged model
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-merged/model.safetensors:   0%|          | 23.9MB / 7.64GB            

CommitInfo(commit_url='https://huggingface.co/viv2005ek/phi-rewriter/commit/c856f6d6b361557c35f353e000b5e2a45e7ecb33', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c856f6d6b361557c35f353e000b5e2a45e7ecb33', pr_url=None, repo_url=RepoUrl('https://huggingface.co/viv2005ek/phi-rewriter', endpoint='https://huggingface.co', repo_type='model', repo_id='viv2005ek/phi-rewriter'), pr_revision=None, pr_num=None)